---
  title: "Aula extra - Zero-shot"
  author: Qiusheng Wu
  date: 2026-21-10
  categories: [GeoAI, technology]
  image: geoai-py.png
  description: "Notebook extraído do github do livro do prof. Quisheng Wu"
---

In [ ]:


import numpy as np
import rasterio
from rasterio.windows import Window
import torch
from transformers import SegformerImageProcessor, AutoModelForSemanticSegmentation

# 1. Carregar modelo
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "nvidia/segformer-b0-finetuned-ade-512-512"

processor = SegformerImageProcessor.from_pretrained(model_id, do_resize=False)
model = AutoModelForSemanticSegmentation.from_pretrained(model_id).to(device)
model.eval()

# 2. Leitura dos canais RGB e normalização do UInt16
tif_path = ":imagem_entrada.tif"

with rasterio.open(tif_path) as src:
    profile = src.profile
    # Lendo bandas 1, 2 e 3 (R, G, B) - descartando NIR no passe RGB
    # Se a ordem do seu arquivo for B, G, R, NIR ajuste a lista de bandas:
    raw_rgb = src.read([1, 2, 3]).astype(np.float32)

# Transpor para (H, W, C)
rgb = np.transpose(raw_rgb, (1, 2, 0))

# Normalização percentil (2% a 98%) para contornar ruídos de nuvens/sombras e converter para uint8
p2, p98 = np.percentile(rgb, (2, 98))
rgb_uint8 = np.clip((rgb - p2) / (p98 - p2 + 1e-5) * 255.0, 0, 255).astype(np.uint8)

height, width, _ = rgb_uint8.shape

# 3. Janela Deslizante (Tiling 512x512 com overlap)
tile_size = 512
stride = 384  # Sobreposição de 128 px para eliminar artefatos nas bordas
num_classes = model.config.num_labels

# Tensores acumuladores para somar logits nas sobreposições
full_logits = np.zeros((num_classes, height, width), dtype=np.float32)
count_map = np.zeros((height, width), dtype=np.float32)

for y in range(0, height, stride):
    for x in range(0, width, stride):
        # Ajustar para não extrapolar a borda final
        y_end = min(y + tile_size, height)
        x_end = min(x + tile_size, width)
        y_start = max(0, y_end - tile_size)
        x_start = max(0, x_end - tile_size)

        patch = rgb_uint8[y_start:y_end, x_start:x_end, :]

        # Inferência no patch
        inputs = processor(images=patch, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits

        # Interpolação para o tamanho exato do tile (512x512)
        patch_logits = torch.nn.functional.interpolate(
            logits,
            size=(tile_size, tile_size),
            mode="bilinear",
            align_corners=False
        ).squeeze(0).cpu().numpy()

        # Acumular logits
        full_logits[:, y_start:y_end, x_start:x_end] += patch_logits
        count_map[y_start:y_end, x_start:x_end] += 1.0

# Média ponderada dos logits nas zonas de sobreposição e geração da máscara final
full_logits /= np.maximum(count_map, 1.0)
final_mask = np.argmax(full_logits, axis=0).astype(np.uint8)

# 4. Gravação mantendo georreferenciamento nativo
output_tif = "segmentacao_tiling_nativo.tif"
profile.update(dtype=rasterio.uint8, count=1, nodata=0)

with rasterio.open(output_tif, "w", **profile) as dst:
    dst.write(final_mask, 1)

print(f"Segmentação em resolução total concluída: {output_tif}")

<positron-console-cell-1>:16: SyntaxWarning: invalid escape sequence '\G'


preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

C:\Users\mauro\.conda\envs\geoai\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mauro\.cache\huggingface\hub\models--nvidia--segformer-b0-finetuned-ade-512-512. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/6.88k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 15.0MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

Segmentação em resolução total concluída: segmentacao_tiling_nativo.tif
